##Imports

In [0]:
from pyspark.sql import functions as F

##Parameters

In [0]:
CATALOG = "workspace"
SCHEMA = "final"

TARGET_TABLE = f"{CATALOG}.{SCHEMA}.gold_dim_date"

##Determine Date Range

In [0]:
sales_df = spark.table(
    "workspace.final.silver_sales_order_header"
)

In [0]:
date_range = sales_df.select(
    F.min("OrderDate").alias("min_date"),
    F.max("OrderDate").alias("max_date")
).collect()[0]

min_date = date_range["min_date"]
max_date = date_range["max_date"]

##Create Calendar DataFrame

In [0]:
date_df = spark.sql(f"""
SELECT explode(
    sequence(
        to_date('{min_date}'),
        to_date('{max_date}'),
        interval 1 day
    )
) AS FullDate
""")

##Create Date Attributes

In [0]:
date_df = (
    date_df

    .withColumn(
        "Date_SK",
        F.date_format(
            F.col("FullDate"),
            "yyyyMMdd"
        ).cast("int")
    )

    .withColumn(
        "Day",
        F.dayofmonth("FullDate")
    )

    .withColumn(
        "Month",
        F.month("FullDate")
    )

    .withColumn(
        "MonthName",
        F.date_format(
            "FullDate",
            "MMMM"
        )
    )

    .withColumn(
        "Quarter",
        F.concat(
            F.lit("Q"),
            F.quarter("FullDate")
        )
    )

    .withColumn(
        "Year",
        F.year("FullDate")
    )

    .withColumn(
        "DayOfWeek",
        F.date_format(
            "FullDate",
            "EEEE"
        )
    )

    .withColumn(
        "WeekOfYear",
        F.weekofyear("FullDate")
    )

    .withColumn(
        "IsWeekend",
        F.when(
            F.dayofweek("FullDate").isin(1,7),
            "Y"
        ).otherwise("N")
    )
)

##Reorder Columns

In [0]:
date_df = date_df.select(
    "Date_SK",
    "FullDate",
    "Day",
    "Month",
    "MonthName",
    "Quarter",
    "Year",
    "DayOfWeek",
    "WeekOfYear",
    "IsWeekend"
)

##Create Table

In [0]:
date_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(TARGET_TABLE)

##Optimize

In [0]:
%sql
OPTIMIZE workspace.final.gold_dim_date

##Validation

In [0]:
%sql
SELECT *
FROM workspace.final.gold_dim_date
ORDER BY FullDate